# Basic example on how to use Enigma

Before getting started, please setup an environment with all the packages installed.

## Mapping model output
FlashRNA uses `TracksMapping` to make it easy to map its multi-head output to corresponding genomic tracks.

In [1]:
from enigma.data.tracks_mapping import TrackDatasetMapping
from enigma.config import DATA_DIR

# Example tracks mapping files
tracks_mapping_path = DATA_DIR / "tracks_mapping_hg38.csv"

# Default is to load only rna, dnase, and atac tracks
tracks_mapping = TrackDatasetMapping(metadata_file=tracks_mapping_path)
print(tracks_mapping)

TrackDatasetMapping(loaded_tracks: ['rna', 'dnase', 'atac'], num_tracks: 1136)


In [2]:
from enigma.data.tracks_mapping import TrackDatasetMapping
from enigma.config import DATA_DIR

# Example tracks mapping files
tracks_mapping_hg38_path = DATA_DIR / "tracks_mapping_hg38.csv"
tracks_mapping_mm10_path = DATA_DIR / "tracks_mapping_mm10.csv"

# Default is to load only rna, dnase, and atac tracks
tracks_mapping = TrackDatasetMapping(metadata_file=tracks_mapping_path)
print(tracks_mapping)

# Load all four types including junction tracks
tracks_mapping = TrackDatasetMapping(metadata_file=tracks_mapping_path, tracks_to_load=["rna", "junction", "dnase", "atac"])
print(tracks_mapping)

# mm10 only has rna, dnase, and atac tracks (no junction tracks)
tracks_mapping = TrackDatasetMapping(metadata_file=tracks_mapping_mm10_path)
print(tracks_mapping)

# The underlying metadata DataFrame
tracks_mapping.metadata_df.head(5)

TrackDatasetMapping(loaded_tracks: ['rna', 'dnase', 'atac'], num_tracks: 1136)
TrackDatasetMapping(loaded_tracks: ['rna', 'junction', 'dnase', 'atac'], num_tracks: 1190)
TrackDatasetMapping(loaded_tracks: ['rna', 'dnase', 'atac'], num_tracks: 258)


,original_index,track_index,identifier,track_type,strand,strand_pair,ontology,gtex_tissue,source,transform_scale,transform_threshold,scale,apply_squashing,pre_transform_scale
0,0,0,CL:0000037 total RNA-seq,rna,+,71,CL:0000037,NaN,encode,1.0,10.0,0.5,True,1.0
1,1,1,CL:0000038 total RNA-seq,rna,+,72,CL:0000038,NaN,encode,1.0,10.0,0.5,True,1.0
2,2,2,CL:0000049 total RNA-seq,rna,+,73,CL:0000049,NaN,encode,1.0,10.0,0.5,True,1.0
3,3,3,CL:0000050 polyA plus RNA-seq,rna,+,74,CL:0000050,NaN,encode,1.0,10.0,0.5,True,1.0
4,4,4,CL:0000050 total RNA-seq,rna,+,75,CL:0000050,NaN,encode,1.0,10.0,0.5,True,1.0


## Initializing a model

In [3]:
from enigma.models import Enigma, EnigmaConfig

### From pre-trained model
We currently have pre-trained Enigma hosted on Wandb as Artifacts.

In [4]:
model = Enigma.from_ckpt(wandb_artifact="deep-genomics-open-source/enigma/distilled-model:v0")

Loading from wandb artifact: deep-genomics-open-source/enigma/distilled-model:v0
Loading model from model.ckpt (loading weights: True)


To load from local:
```python
model = Enigma.from_ckpt(ckpt_path="path/to/checkpoint.ckpt")
```

### Creating from scratch

In [5]:
# From scratch -- with default config settings
# First, need to provide tracks mapping
tracks_mapping = {
    "hg38": TrackDatasetMapping(metadata_file=tracks_mapping_hg38_path, tracks_to_load=["rna", "junction", "dnase", "atac"]),
    "mm10": TrackDatasetMapping(metadata_file=tracks_mapping_mm10_path, tracks_to_load=["rna", "dnase", "atac"]),
}
config = EnigmaConfig(tracks_mapping=tracks_mapping)
model_scratch_1 = Enigma(config=config)
print(f"Default config: {model_scratch_1.config}")

# From scratch -- changing the model architecture
config = EnigmaConfig(tracks_mapping=tracks_mapping, transformer_num_layers=4)
model_scratch_2 = Enigma(config=config)
print(f"Smaller model: {model_scratch_2.config}")

Default config: EnigmaConfig(model_type='enigma', species='multi-species', tracks_mapping={'hg38': TrackDatasetMapping(loaded_tracks: ['rna', 'junction', 'dnase', 'atac'], num_tracks: 1190), 'mm10': TrackDatasetMapping(loaded_tracks: ['rna', 'dnase', 'atac'], num_tracks: 258)}, prediction_resolution=1, prediction_crop_margin=None, dim=1536, transformer_num_layers=8, head_dim=192, expansion_factor=2, transformer_norm='ln', use_rope=True, rope_base=10000.0, use_alibi=False, window_size=(-1, -1), mlp_activation='gelu_tanh', mlp_swiglu_match_params=False, mlp_dropout=0.3, attention_dropout=0.2, post_attn_dropout=0.3, post_mlp_dropout=0.3, use_qk_norm=True, conv_num_layers=0, conv_dim_in=256, conv_dim_out=256, conv_kernel_size=5, conv_activation='gelu_tanh', conv_norm='group_32', stem_dropout=0.0, unet_dim_out=768, unet_num_downsampling=7, encoder_kernel_size=5, decoder_kernel_size=3, unet_activation='gelu_tanh', unet_norm='group_32', unet_dropout=0.0, embedding_conv_kernel_size=15, output_

## Basic Predictions

In this example, we create an input sequence using `GenomeKit`. Please checkout https://github.com/deepgenomics/GenomeKit if you are interested in learning more!

Please install `GenomeKit` before moving forward.
```bash
conda install genomekit
```

In [6]:
from genome_kit import Genome, Interval

# Creating an input sequence
genome = Genome('hg38')
itv = Interval('chr4', '+', 82360581, 82884869, genome)
sequence = genome.dna(itv)  # this fetches genomic sequence corresponding to the interval

/home/andrew/miniforge3/envs/enigma_internal/lib/python3.12/site-packages/google/cloud/storage/__init__.py:35: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution
/home/andrew/miniforge3/envs/enigma_internal/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
# First, prep the model for inference
model.to('cuda')
model.eval()
assert not model.training

In [8]:
from enigma.data.sequences import Sequence  # helper class

input_1 = Sequence.from_string(sequence).to('cuda')

Important: FlashAttention requires bf16 or fp16 and FlashRNA is trained with bf16 mixed-precision. 

For more details on FlashAttention: https://github.com/Dao-AILab/flash-attention

In [9]:
import torch

with torch.autocast(device_type='cuda', dtype=torch.bfloat16), torch.no_grad():
    preds_1 = model.predict(input_1)

In [10]:
# Model prediction class has some helper functions
print(f"Predictions class: {type(preds_1)}")

# Get just the predictions tensor for the rna track
preds_1.get_tracks('rna')

# Get the tracks mapping
preds_1.tracks_mapping

# To get predictions for GTEx "Stomach" tissue
indices = preds_1.tracks_mapping.indices_by_gtex_rna("Stomach")
preds_tensor_for_stomach = preds_1.tracks[..., indices] # [b, l, d]

Predictions class: <class 'enigma.data.seqtrack.TargetTracks'>


In [11]:
# Predictions with reverse complement augmentation
with torch.autocast(device_type='cuda', dtype=torch.bfloat16), torch.no_grad():
    preds_2 = model.predict(input_1, pred_reverse_complement=True)

In [12]:
# Underneath, RCSequences helps creating a batch with reverse complement sequences
from enigma.data.sequences import RCSequences

input_2 = RCSequences.from_string(sequence).to('cuda')
print(f"Input 1: {input_1.tensor.shape}")
print(f"Input 2: {input_2.tensor.shape}")

# Can use RCSequences directly
with torch.autocast(device_type='cuda', dtype=torch.bfloat16), torch.no_grad():
    preds_3 = model.predict(input_2, pred_reverse_complement=True)

assert torch.isclose(preds_2.tracks, preds_3.tracks).all()

Input 1: torch.Size([1, 524288])
Input 2: torch.Size([2, 524288])


In [13]:
# Can also use PyTorch Tensors directly
with torch.autocast(device_type='cuda', dtype=torch.bfloat16), torch.no_grad():
    preds_4 = model.predict(input_1.tensor)

assert torch.isclose(preds_1.tracks, preds_4.tracks).all()